# RNN 
- 2 Entradas PwmD, PwmE
- 1 Saida $\Delta \theta $ 
- 1 Camada
- n neurônios

In [9]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from keras import initializers

Datasets = []
PREDICTORS = ["PwmD", "PwmE"]   
TARGET_INT = ["Wd", "We"]  
TARGET = ["Wd", "We"]
     
TIME_STEPS = 3
TS = 0.07

In [10]:
for i in range(4):
    Dataset = pd.read_excel(f"../../../../RotedData/Data.xlsx", f"D{i+1}")   
    Datasets.append(Dataset)

for i in range(2):   
    Dataset = pd.read_csv(f"../../../../Data/Data{i + 1}.csv")  
    Datasets.append(Dataset)
    
    
for i in range(len(Datasets)):
    Dataset = Datasets[i].copy()

    # for var in TARGET_INT:
    #     Dataset[f"Delta{var}"] = (Dataset[var].shift(-1) - Dataset[var]) / TS

    # Dataset = Dataset.dropna(subset=[f"Delta{var}" for var in TARGET_INT])

    Datasets[i] = Dataset

In [11]:
NormDatasets = []

SCALER = StandardScaler()
OUT_SCALER = StandardScaler()

Train1 = Datasets[0].copy()
Train1[PREDICTORS] = SCALER.fit_transform(Train1[PREDICTORS])
Train1[TARGET] = OUT_SCALER.fit_transform(Train1[TARGET])
NormDatasets.append(Train1)

Train2 = Datasets[1].copy()
Train2[PREDICTORS] = SCALER.transform(Train2[PREDICTORS])
Train2[TARGET] = OUT_SCALER.transform(Train2[TARGET])
NormDatasets.append(Train2)

# concatena
Train = pd.concat([Train1, Train2], ignore_index=True)

for i in range(4):
    CurrentTestDataset = Datasets[i + 2].copy()
    CurrentTestDataset[PREDICTORS] = SCALER.transform(CurrentTestDataset[PREDICTORS])
    CurrentTestDataset[TARGET] = OUT_SCALER.transform(CurrentTestDataset[TARGET])
    NormDatasets.append(CurrentTestDataset)

In [12]:
def CreateSequences(input_data, target_data, timesteps):
    X_seq, Y_seq = [], []
    
    for i in range(timesteps, len(input_data)):
        X_seq.append(input_data.iloc[i-timesteps:i].values)
        Y_seq.append(target_data.iloc[i])
    return np.array(X_seq), np.array(Y_seq)

In [13]:
x_train, y_train = CreateSequences(Train[PREDICTORS], Train[TARGET], TIME_STEPS)

x_val, y_val = CreateSequences((NormDatasets[5])[PREDICTORS], (NormDatasets[5])[TARGET], TIME_STEPS)
print(f"Dimensão da entrada: {np.shape(x_train)}")
print(f"Dimensão da saida: {np.shape(y_train)}")

print(f"Dimensão da entrada: {np.shape(x_val)}")
print(f"Dimensão da saida: {np.shape(y_val)}")

Dimensão da entrada: (1951, 3, 2)
Dimensão da saida: (1951, 2)
Dimensão da entrada: (1269, 3, 2)
Dimensão da saida: (1269, 2)


In [14]:
import matplotlib.pyplot as plt

def PlotHistory(history):
    plt.figure(figsize=(8, 5))
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('MSE Loss')
    plt.title('Training History')
    plt.legend()
    plt.grid(True)
    plt.show()

In [15]:
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import numpy as np

TITLES = ["train1", "train2", "test1", "test2", "test3", "val"]

def PlotOut(ax, title, target_name, y_true, y_pred):
    time = (np.arange(0, len(y_pred), 1).astype(float) * 0.07).round(5)

    ax.scatter(time, y_true, marker='o', s=12, label='Amostras Reais', alpha=0.7)
    ax.scatter(time, y_pred, marker='x', s=12, label='Valores Preditos', alpha=0.7)
    ax.set_title(f'{title} - {target_name}')
    ax.set_xlabel('Tempo [s]')
    ax.set_ylabel(target_name)
    ax.legend()
    ax.grid(True)


def EvalModel(model):
    n_targets = len(TARGET)

    metrics = {name: {"R2_train1": [], "R2_train2": [],
                      "R2_test1": [], "R2_test2": [],
                      "R2_test3": [],"R2_val": [],
                      "MSE_train1": [], "MSE_train2": [],
                      "MSE_test1": [], "MSE_test2": [],
                      "MSE_test3": [], "MSE_val": [],} for name in TARGET_INT}

    for i, dataset in enumerate(NormDatasets):
        x = dataset[PREDICTORS]
        y = dataset[TARGET]
        x, y = CreateSequences(x, y, TIME_STEPS)

        y_pred = OUT_SCALER.inverse_transform(model.predict(x))
        y_true = OUT_SCALER.inverse_transform(y)
        
        # Calcula métricas por saída
        for j, name in enumerate(TARGET_INT):
            r2 = r2_score(y_true[:, j], y_pred[:, j])
            mse = mean_squared_error(y_true[:, j], y_pred[:, j])
            metrics[name][f"R2_{TITLES[i]}"].append(r2)
            metrics[name][f"MSE_{TITLES[i]}"].append(mse)

            print(f"{name} | {TITLES[i]} -> R² = {r2:.4f}, MSE = {mse:.4e}")
            
    # Retorna métricas médias para análise posterior
    return metrics

In [16]:
import numpy as np
import pandas as pd
from tensorflow import keras
from keras.callbacks import EarlyStopping
from keras import initializers
from keras import regularizers
from itertools import product

# tamanho da entrada
INPUT_SIZE = len(PREDICTORS)  
OUTPUT_SIZE = len(TARGET) 
N_MODELS = 5  # número de inicializações

seeds = np.random.choice(range(1, 10000), size=N_MODELS, replace=False)

# arquiteturas das camadas ocultas
architectures = [
    [32], [64], [128], [264],
    [8, 4], [16, 8], [32, 16], [64, 32], [128, 64], [264, 128],
    [16, 8, 4], [32, 16, 8], [128, 64, 32], [264, 128, 64]
]
r_values = [0.01, 0.5, 0.9]

results = {}
excel_file = "resultados.xlsx"

for arch,  r in product(architectures, r_values):
    for i, s in enumerate(seeds):

        initializer = initializers.RandomNormal(seed=int(s))
        regularizer = regularizers.l2(r)

        model = keras.models.Sequential()
        model.add(keras.layers.Input(shape=(TIME_STEPS, INPUT_SIZE)))

        # adiciona camadas RNN
        for j, n in enumerate(arch):

            # se não for a última camada RNN
            if j < len(arch) - 1:
                model.add(
                    keras.layers.SimpleRNN(
                        n,
                        activation="tanh",
                        return_sequences=True,
                        kernel_initializer=initializer,
                        kernel_regularizer=regularizer,
                        bias_regularizer=regularizer
                    )
                )
            else:
                model.add(
                    keras.layers.SimpleRNN(
                        n,
                        activation="tanh",
                        kernel_initializer=initializer,
                        kernel_regularizer=regularizer,
                        bias_regularizer=regularizer
                    )
                )

        # camada de saída
        model.add(
            keras.layers.Dense(
                OUTPUT_SIZE,
                activation="linear",
                kernel_initializer=initializer,
                kernel_regularizer=regularizer,
                bias_regularizer=regularizer
            )
        )
        # salva pesos iniciais
        w0 = model.get_weights()

        # compila
        model.compile(loss="mean_squared_error", optimizer="adam")
        early_stopping_monitor = EarlyStopping(
            monitor='val_loss',
            patience=50,
            restore_best_weights=True
        )

        history = model.fit(
            x_train, 
            y_train, 
            epochs=1000,
            callbacks=[early_stopping_monitor],
            validation_data=(x_val, y_val),
            verbose=0
        )
        
    
        # salva pesos finais
        wf = model.get_weights()

       # avalia o modelo
        metrics = EvalModel(model)

        arch_name = "_".join(map(str, arch))

        row = {
            "model": f"model_{arch_name}_{i}",
            "Neurons": arch,
            "W0": str([w.round(4).tolist() for w in w0]),
            "Wf": str([w.round(4).tolist() for w in wf]),
        }


        for name in TARGET_INT:
            row.update({
                f"R2_Train_1_{name}": metrics[name]["R2_train1"],
                f"MSE_Train_1_{name}": metrics[name]["MSE_train1"],
                f"R2_Train_2_{name}": metrics[name]["R2_train2"],
                f"MSE_Train_2_{name}": metrics[name]["MSE_train2"],
                f"R2_Val_{name}": metrics[name]["R2_val"],
                f"MSE_Val_{name}": metrics[name]["MSE_val"],
                f"R2_Test_1_{name}": metrics[name]["R2_test1"],
                f"MSE_Test_1_{name}": metrics[name]["MSE_test1"],
                f"R2_Test_2_{name}": metrics[name]["R2_test2"],
                f"MSE_Test_2_{name}": metrics[name]["MSE_test2"],
                f"R2_Test_3_{name}": metrics[name]["R2_test3"],
                f"MSE_Test_3_{name}": metrics[name]["MSE_test3"],
            })
        
        df = pd.DataFrame([row])

        # salva/atualiza Excel incrementalmente
        try:
            # tenta abrir existente e adicionar linha
            old = pd.read_excel(excel_file)
            new_df = pd.concat([old, df], ignore_index=True)
            new_df.to_excel(excel_file, index=False)
        except FileNotFoundError:
            # se não existir, cria arquivo novo
            df.to_excel(excel_file, index=False)

        print(f"Modelo {i} treinado e salvo no Excel")

31/31 [==============================] - 0s 2ms/step
Wd | train1 -> R² = 0.9213, MSE = 4.1541e-01
We | train1 -> R² = 0.9023, MSE = 5.3105e-01
31/31 [==============================] - 0s 2ms/step
Wd | train2 -> R² = 0.9139, MSE = 4.5292e-01
We | train2 -> R² = 0.9167, MSE = 4.5242e-01
31/31 [==============================] - 0s 2ms/step
Wd | test1 -> R² = 0.9085, MSE = 4.7516e-01
We | test1 -> R² = 0.8951, MSE = 5.4732e-01
31/31 [==============================] - 0s 2ms/step
Wd | test2 -> R² = 0.9173, MSE = 4.2564e-01
We | test2 -> R² = 0.9324, MSE = 3.5552e-01
45/45 [==============================] - 0s 2ms/step
Wd | test3 -> R² = 0.8206, MSE = 6.6844e-01
We | test3 -> R² = 0.7933, MSE = 7.4110e-01
40/40 [==============================] - 0s 2ms/step
Wd | val -> R² = 0.8468, MSE = 6.4968e-01
We | val -> R² = 0.8411, MSE = 6.6978e-01
Modelo 0 treinado e salvo no Excel
31/31 [==============================] - 0s 3ms/step
Wd | train1 -> R² = 0.9247, MSE = 3.9749e-01
We | train1 -> R² = 0